In [ ]:
import os
from tqdm import tqdm
import numpy as np
from moviepy import VideoFileClip
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import Video

notebook_dir = os.getcwd()
data_dir = Path(notebook_dir).parents[1]/'data_samples'

# Have to have conda bin on path, so ffmpeg can be called.
os.environ["PATH"] += os.path.pathsep + "/work/scratch-nopw2/bantonio/iecdt_eda_env/bin"

First lets load the mp4 file using Moviepy. We can extract the audio into a separate object, using the .audio property of the video clip object

In [ ]:
video_file = data_dir/'audio_visual'/'video_with_sound_example.mp4'
video_clip = VideoFileClip(str(video_file))
audio_clip = video_clip.audio

In [ ]:
# Play the video clip
video_clip.write_videofile("temp_output.mp4", fps=30)

Video("temp_output.mp4", width=500)

In [ ]:
# Using .iter_framse we can iterate through the video clips
# As an example, lets calculate the maximum intensity at each frame
averages = []
for frame in tqdm(audio_clip.iter_frames()):
    
    averages.append(frame.max())
plt.plot(averages)

In [ ]:
# Length of video clip
video_clip.duration

In [ ]:
# Frames per second
video_clip.fps

In [ ]:
dir(video_clip.reader)

In [ ]:
# Number of frames
video_clip.reader.n_frames

In [ ]:
# Selecting frames by time (in seconds)
frame = video_clip.get_frame(5)

# Extract frame by index; moveipy doesn't seem to support this natively, so you have to use the frames-per-second property
frame = video_clip.get_frame(100 / video_clip.fps)

In [ ]:
# Similar methods work for audio clips
audio_clip.get_frame(1)

## Loading Audio files

In [ ]:
wav_filename = str(data_dir/'audio_visual'/'audio_example.wav')

There are many tools available to process audio

In [ ]:
# the most basic way of loading wav files is using scipy.io
from scipy.io import wavfile

vsample_rate, audio_data = wavfile.read(os.path.join(data_dir, 'audio_visual', 'audio_example.wav'))

# The audio data is a numpy array, with dimensions [N_samples, N_channels]
# In this case there are two channels, with near identical signals in each

print('Audio Shape:',np.shape(audio_data))

In [ ]:
# For situations where you require more advanced audio manipulations, pydub is also a useful package, and can load many more file formats than scipy.io
from pydub import AudioSegment

pydub_clip = AudioSegment.from_wav(wav_filename)


In [ ]:
# pydub also has a nice integration with Jupyter notebooks 
pydub_clip

In [ ]:
# Converting from pydub to numpy is not as easy as it should be...
# But we can use this function from https://stackoverflow.com/questions/38015319/how-to-create-a-numpy-array-from-a-pydub-audiosegment

def pydub_to_np(audio: AudioSegment) -> tuple[np.ndarray, int]:
    """
    Converts pydub audio segment into np.float32 of shape [duration_in_seconds*sample_rate, channels],
    where each value is in range [-1.0, 1.0]. 
    Returns tuple (audio_np_array, sample_rate).
    """
    return np.array(audio.get_array_of_samples(), dtype=np.float32).reshape((-1, audio.channels)) / (
            1 << (8 * audio.sample_width - 1)), audio.frame_rate

In [ ]:
# Convert the pydub clip to a numpy array (still in stereo)
numpy_stereo_data, frame_rate = pydub_to_np(pydub_clip)

In [ ]:
# Note the dimensions; 2 channels since the signal is in stereo
numpy_stereo_data.shape

In [ ]:
# Plot part of the audio signal, note the two different channels for left and right speaker

fig, ax = plt.subplots(2,1, figsize=(5,10))
ax[0].plot(numpy_stereo_data[:5000,0], 'r')
ax[0].set_title('Left')

ax[1].plot(numpy_stereo_data[:5000,1], 'g')
ax[1].set_title('Right')

In [ ]:
# Change from stereo (2 channels) to mono (1 channel)
mono_pydub_clip = pydub_clip.set_channels(1) 

numpy_mono_data, frame_rate = pydub_to_np(mono_pydub_clip)